# Search-02d : RRT et RRT* en C#

[Version Python](Search-02d-RRT-Recherche-Continue.ipynb) | [Index Search](../README.md)

Un robot ponctuel explore un carré avec obstacles. Nous comparons RRT et une variante pédagogique de RRT* : sélection du meilleur parent et reparentage. Durée : environ 45 minutes. Prérequis : géométrie plane et parcours d'arbre. Les générateurs aléatoires Python et .NET diffèrent : une même graine ne garantit pas les mêmes points.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine("RRT : collections et géométrie BCL");

RRT : collections et géométrie BCL


## 1. Terrain et collisions

Le carré est $[0,10]^2$. On teste exactement l’intersection des segments avec les rectangles fermés. Les bords sont considérés bloqués.

In [2]:
public record struct Point(double X, double Y);
public record struct Rectangle(double X0, double Y0, double X1, double Y1);
public class Terrain {
    public List<Rectangle> Obstacles;
    public Terrain(params Rectangle[] obstacles) { Obstacles = obstacles.ToList(); }
    public bool Bloque(Point p) => p.X < 0 || p.X > 10 || p.Y < 0 || p.Y > 10 ||
        Obstacles.Any(r => r.X0 <= p.X && p.X <= r.X1 && r.Y0 <= p.Y && p.Y <= r.Y1);
    // Intersection par dalles, rectangles fermés et dégénérés inclus.
    public bool SegmentBloque(Point a, Point b) {
        if(Bloque(a)||Bloque(b)) return true;
        foreach(var r in Obstacles) {
            double lo=0,hi=1; bool possible=true;
            foreach(var (origine,delta,bas,haut) in new[]{(a.X,b.X-a.X,r.X0,r.X1),(a.Y,b.Y-a.Y,r.Y0,r.Y1)}) {
                if(delta==0) { if(origine<bas||origine>haut) {possible=false;break;} }
                else { double t0=(bas-origine)/delta,t1=(haut-origine)/delta; lo=Math.Max(lo,Math.Min(t0,t1)); hi=Math.Min(hi,Math.Max(t0,t1)); if(lo>hi) {possible=false;break;} }
            }
            if(possible) return true;
        }
        return false;
    }
}
public static class Geo {
    public static double Distance(Point a, Point b) => Math.Sqrt(Math.Pow(a.X-b.X,2)+Math.Pow(a.Y-b.Y,2));
    public static double Longueur(IReadOnlyList<Point> chemin) => Enumerable.Range(1,chemin.Count-1).Sum(i => Distance(chemin[i-1],chemin[i]));
    public static Point Avancer(Point a, Point b, double pas) { double d=Distance(a,b); return d<1e-9 ? a : new Point(a.X+Math.Min(pas,d)*(b.X-a.X)/d,a.Y+Math.Min(pas,d)*(b.Y-a.Y)/d); }
}
var terrain = new Terrain(new Rectangle(2,1,2,6), new Rectangle(4.5,4,7.5,6.5), new Rectangle(7,2,8.5,6.5));
var depart = new Point(0.5,9.5); var but = new Point(9.5,0.5);
Console.WriteLine($"Borne inférieure : {Geo.Distance(depart,but):F2}");
if (!terrain.SegmentBloque(new Point(1,3),new Point(3,3))) throw new Exception("Mur traversé");

Borne inférieure : 12.73


### Exercice 1 — plus proche

Complétez une recherche en une passe. Le stub retourne `null` afin de permettre l'exécution ; le contrôle accepte cet état ou la réponse attendue.

In [3]:
int? PlusProcheExercice(Point p, IReadOnlyList<Point> points) {
    // TODO : retourner l'indice le plus proche, ou null si la liste est vide.
    return null;
}
var e1 = PlusProcheExercice(new Point(0.1,0), new[]{new Point(0,0),new Point(5,0)});
if (e1 != null && e1 != 0) throw new Exception("Exercice 1");
Console.WriteLine(e1?.ToString() ?? "Exercice 1 à compléter");

Exercice 1 à compléter


## 2. RRT et RRT*

RRT rattache un point dirigé vers l'échantillon à son plus proche voisin. RRT* cherche les candidats autour du **nouveau point**, filtre les segments bloqués et choisit le coût minimal. Lors d'un reparentage, la même variation de coût s'applique à tous les descendants. La meilleure connexion au but est recherchée à la fin du budget afin de tenir compte des améliorations tardives.

In [4]:
public class Noeud {
    public Point P; public int Parent; public double Cout; public HashSet<int> Enfants = new();
    public Noeud(Point p,int parent,double cout) { P=p; Parent=parent; Cout=cout; }
}
public record Resultat(List<Point> Chemin, List<Noeud> Noeuds);
public static class Planificateur {
    public static List<int> Voisins(List<Noeud> ns, Point q, double rho) => Enumerable.Range(0,ns.Count).Where(i=>Geo.Distance(ns[i].P,q)<=rho).ToList();
    public static int ChoisirParent(List<Noeud> ns, IEnumerable<int> candidats, Point q, Terrain t) =>
        candidats.Where(k=>!t.SegmentBloque(ns[k].P,q)).MinBy(k=>ns[k].Cout+Geo.Distance(ns[k].P,q));
    public static void Reparenter(List<Noeud> ns, int y, int u) {
        double nouveau=ns[u].Cout+Geo.Distance(ns[u].P,ns[y].P), delta=nouveau-ns[y].Cout;
        ns[ns[y].Parent].Enfants.Remove(y); ns[y].Parent=u; ns[u].Enfants.Add(y);
        var pile=new Stack<int>(); pile.Push(y);
        while(pile.Count>0) { int v=pile.Pop(); ns[v].Cout+=delta; foreach(int enfant in ns[v].Enfants) pile.Push(enfant); }
    }
    public static Resultat Plan(Terrain t, Point depart, Point but, int budget, int graine, bool etoile, double pas=0.5, double rho=1.5, double biais=0.1, Action<List<Noeud>> apresRewire=null) {
        var rng=new Random(graine); var ns=new List<Noeud>{new Noeud(depart,-1,0)}; var buts=new List<int>();
        for(int iteration=0; iteration<budget; iteration++) {
            var cible=rng.NextDouble()<biais ? but : new Point(10*rng.NextDouble(),10*rng.NextDouble());
            int proche=Enumerable.Range(0,ns.Count).MinBy(k=>Geo.Distance(ns[k].P,cible));
            if(Geo.Distance(ns[proche].P,cible)<1e-9) continue;
            var q=Geo.Avancer(ns[proche].P,cible,pas);
            if(t.Bloque(q)||t.SegmentBloque(ns[proche].P,q)) continue;
            var voisins=etoile ? Voisins(ns,q,rho) : new List<int>();
            if(!voisins.Contains(proche)) voisins.Add(proche);
            int u=ChoisirParent(ns,voisins,q,t);
            int nouveau=ns.Count; ns.Add(new Noeud(q,u,ns[u].Cout+Geo.Distance(ns[u].P,q))); ns[u].Enfants.Add(nouveau);
            if(etoile) foreach(int y in voisins) {
                if(ns[nouveau].Cout+Geo.Distance(q,ns[y].P)<ns[y].Cout-1e-12 && !t.SegmentBloque(q,ns[y].P)) { Reparenter(ns,y,nouveau); apresRewire?.Invoke(ns); }
            }
            if(Geo.Distance(q,but)<0.3 && !t.SegmentBloque(q,but)) buts.Add(nouveau);
        }
        List<Point> chemin=null;
        if(buts.Count>0) { int j=buts.MinBy(k=>ns[k].Cout+Geo.Distance(ns[k].P,but)); chemin=new List<Point>{but}; while(j!=-1){chemin.Add(ns[j].P);j=ns[j].Parent;} chemin.Reverse(); }
        return new Resultat(chemin,ns);
    }
}
Console.WriteLine("Planificateurs définis : pas 0,5, rayon 1,5, biais 0,1.");


Planificateurs définis : pas 0,5, rayon 1,5, biais 0,1.


### Contrôles indépendants

On reconstruit le coût de chaque nœud en remontant ses parents, en vérifiant l'absence de cycles et chaque segment. Le contre-exemple vérifie aussi la propagation vers un descendant et le centre de la boule.

In [5]:
// Contrôle indépendant : intersections d'arêtes par orientations, pas par dalles.
bool CheminValide(List<Point> ch, Point d, Point b, Terrain t) {
    if(ch==null || ch.Count==0 || ch[0]!=d || ch[^1]!=b || ch.Any(t.Bloque)) return false;
    double O(Point a,Point b,Point c)=>(b.X-a.X)*(c.Y-a.Y)-(b.Y-a.Y)*(c.X-a.X);
    bool Sur(Point a,Point b,Point p)=>Math.Min(a.X,b.X)<=p.X && p.X<=Math.Max(a.X,b.X) && Math.Min(a.Y,b.Y)<=p.Y && p.Y<=Math.Max(a.Y,b.Y);
    bool Croise(Point a,Point b,Point c,Point d) {
        double o1=O(a,b,c),o2=O(a,b,d),o3=O(c,d,a),o4=O(c,d,b);
        return (o1==0&&Sur(a,b,c))||(o2==0&&Sur(a,b,d))||(o3==0&&Sur(c,d,a))||(o4==0&&Sur(c,d,b))||(o1*o2<0 && o3*o4<0);
    }
    for(int i=1;i<ch.Count;i++) foreach(var r in t.Obstacles) {
        Point[] c={new(r.X0,r.Y0),new(r.X1,r.Y0),new(r.X1,r.Y1),new(r.X0,r.Y1)};
        for(int k=0;k<4;k++) if(Croise(ch[i-1],ch[i],c[k],c[(k+1)%4])) return false;
    }
    return true;
}
foreach(var rect in new[]{new Rectangle(2,1,2,6),new Rectangle(1.999,1,2.001,6)}) {
    var t=new Terrain(rect); var a=new Point(1.01,3); var b=new Point(3.01,3);
    if(!t.SegmentBloque(a,b)||CheminValide(new List<Point>{a,b},a,b,t)||!t.SegmentBloque(new Point(1,6),new Point(3,6))||t.SegmentBloque(new Point(1,7),new Point(3,7))||!t.SegmentBloque(new Point(2,3),new Point(2,3))) throw new Exception("Collision exacte incorrecte");
}
void Verifier(Resultat r, Terrain t) {
    for(int i=0;i<r.Noeuds.Count;i++) {
        var vus=new HashSet<int>(); int j=i; double somme=0;
        while(j!=0) { if(!vus.Add(j)) throw new Exception("Cycle"); int p=r.Noeuds[j].Parent;
            if(p<0 || p>=r.Noeuds.Count || !CheminValide(new List<Point>{r.Noeuds[j].P,r.Noeuds[p].P},r.Noeuds[j].P,r.Noeuds[p].P,t)) throw new Exception("Arête invalide");
            somme+=Geo.Distance(r.Noeuds[j].P,r.Noeuds[p].P); j=p;
        }
        if(Math.Abs(somme-r.Noeuds[i].Cout)>1e-8) throw new Exception("Coût incohérent");
    }
}
var test=new List<Noeud>{new(new Point(0,0),-1,0),new(new Point(0,2),0,2),new(new Point(2,2),1,4),new(new Point(1,0),0,1),new(new Point(3,2),2,5)};
test[0].Enfants.UnionWith(new[]{1,3}); test[1].Enfants.Add(2); test[2].Enfants.Add(4);
Planificateur.Reparenter(test,2,3); Verifier(new Resultat(null,test),new Terrain());
var points=new List<Noeud>{new(new Point(0,0),-1,0),new(new Point(0.9,0),0,0),new(new Point(1.4,0),0,0)};
var v=Planificateur.Voisins(points,new Point(0.5,0),1);
if(!v.Contains(2) || v.Count!=3) throw new Exception("Voisinage incorrect");
var choix = new List<Noeud>{new(new Point(0,0),-1,0),new(new Point(1.9,0),0,5)};
if(Planificateur.ChoisirParent(choix,new[]{0,1},new Point(2,0),new Terrain())!=0) throw new Exception("Mauvais parent");
int rewires=0;
Planificateur.Plan(terrain,depart,but,300,0,true,apresRewire: ns=>{rewires++; Verifier(new Resultat(null,ns),terrain);});
if(rewires==0) throw new Exception("Aucun reparentage testé");
Console.WriteLine($"Invariants vérifiés après {rewires} reparentages et contre-exemples vérifiés");

Invariants vérifiés après 85 reparentages et contre-exemples vérifiés


### Exercice 2 — meilleur parent

Dans la liste des candidats libres, minimisez coût cumulé plus longueur du segment. Les coûts de l'exemple sont 2 et 5,1.

In [6]:
int? MeilleurParentExercice(Point q, IReadOnlyList<Noeud> ns, IReadOnlyList<int> candidats) {
    // TODO : choisir l'indice minimisant ns[i].Cout + Geo.Distance(ns[i].P,q).
    return null;
}
var candidatsEx=new List<Noeud>{new(new Point(0,0),-1,0),new(new Point(1.9,0),0,5)};
var e2=MeilleurParentExercice(new Point(2,0),candidatsEx,new[]{0,1});
if(e2!=null && e2!=0) throw new Exception("Exercice 2");
Console.WriteLine(e2?.ToString() ?? "Exercice 2 à compléter");

Exercice 2 à compléter


## 3. Comparaison sur trois obstacles

Les tracés montrent une seule graine. Un échec est affiché explicitement ; le ratio compare la longueur à la distance à vol d'oiseau, borne inférieure qui ignore les obstacles.

In [7]:
var rrt=Planificateur.Plan(terrain,depart,but,1500,0,false);
var star=Planificateur.Plan(terrain,depart,but,1500,0,true);
Verifier(rrt,terrain); Verifier(star,terrain);
if(rrt.Chemin!=null && !CheminValide(rrt.Chemin,depart,but,terrain) || star.Chemin!=null && !CheminValide(star.Chemin,depart,but,terrain)) throw new Exception("Chemin invalide");
foreach(var pair in new[]{("RRT",rrt),("RRT*",star)})
    Console.WriteLine(pair.Item2.Chemin==null ? $"{pair.Item1} : échec" : $"{pair.Item1} : {pair.Item2.Noeuds.Count} nœuds, longueur {Geo.Longueur(pair.Item2.Chemin):F2}, ratio {Geo.Longueur(pair.Item2.Chemin)/Geo.Distance(depart,but):F2}");

RRT : 1089 nœuds, longueur 15.83, ratio 1.24


RRT* : 1089 nœuds, longueur 12.96, ratio 1.02


In [8]:
public record PlotSvg(string Content);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<PlotSvg>((v,w)=>w.Write(v.Content), "text/html");
string Dessiner(Terrain t, List<Point> ch, string nom) {
    var s=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Terrain et chemin' viewBox='0 0 450 450' width='450'><rect width='450' height='450' fill='white'/><rect x='10' y='10' width='400' height='400' fill='none' stroke='black'/>");
    foreach(var r in t.Obstacles) {
        if(r.X0==r.X1 || r.Y0==r.Y1) s.Append($"<line x1='{10+40*r.X0}' y1='{410-40*r.Y0}' x2='{10+40*r.X1}' y2='{410-40*r.Y1}' stroke='#444' stroke-width='2'/>");
        else s.Append($"<rect x='{10+40*r.X0}' y='{410-40*r.Y1}' width='{40*(r.X1-r.X0)}' height='{40*(r.Y1-r.Y0)}' fill='#aaa' stroke='#444'/>");
    }
    if(ch!=null) s.Append("<polyline fill='none' stroke='#1656a0' stroke-width='2' points='"+string.Join(" ",ch.Select(p=>$"{10+40*p.X},{410-40*p.Y}"))+"'/>");
    s.Append($"<text x='150' y='445'>{nom} — x (0 à 10)</text><text x='415' y='200'>y</text><text x='10' y='425'>0</text><text x='390' y='425'>10</text><text x='414' y='15'>10</text>");
    s.Append($"<circle cx='{10+40*depart.X}' cy='{410-40*depart.Y}' r='5' fill='green'/><circle cx='{10+40*but.X}' cy='{410-40*but.Y}' r='5' fill='red'/></svg>"); return s.ToString();
}
display(new PlotSvg(Dessiner(terrain,rrt.Chemin,"RRT")));
display(new PlotSvg(Dessiner(terrain,star.Chemin,"RRT*")));
Console.WriteLine("Gauche : RRT ; droite : RRT* (départ vert, but rouge).");

<polyline fill='none' stroke='#1656a0' stroke-width='2' points='30,30 49.46022646390234,34.61514744874256 54.90456529050233,53.85986554855316 74.24057372813137,48.74921219502397 80.01014942312959,67.89893794501018 93.87876836949022,82.30939971834164 113.55968469689437,85.86770444302687 133.51414395024042,84.5187952887278 153.11948491861418,88.47235134799973 162.40030037137228,106.18862817276869 171.66816558585253,123.91168305521569 173.52634143212114,143.82517552952424 176.84415477662975,163.54805854928077 189.26583331253676,179.22293980403313 182.34866906400427,197.98867556788807 188.2743518040832,217.09067242852885 180.43538399526636,216.62572766031406 174.4478536235911,235.70842872825165 183.05448279033772,253.7618476395301 193.21642607229893,270.9878427844906 202.06703482438624,288.9229124726038 210.91530410615513,306.42680226658786 229.54373521825386,299.1479593433073 247.35372882008878,308.247636694873 261.6436086267439,294.25480513327153 277.6748541453967,306.2130310518882 283.47520646971526,325.353457194317 296.69523232347285,340.3611524346255 314.3088165925989,349.83541489639 331.97561180739655,359.21008005512505 349.6424070221942,368.5847452138602 367.30920223699184,377.95941037259524 384.9759974517895,387.33407553133037 390,390'/> RRT — x (0 à 10) y 0 10 10

RRT* — x (0 à 10) y 0 10 10

Gauche : RRT ; droite : RRT* (départ vert, but rouge).


## 4. Budgets et graines

Le mur laisse un passage en bas. Pour chaque budget et chaque graine, on indique la médiane des ratios parmi les succès **et leur nombre**. Ces mesures finies ne démontrent pas l'optimalité asymptotique : le théorème du RRT* repose sur des hypothèses qui ne sont pas établies pour cette variante à rayon fixe et connexion au but particulière.

In [9]:
var mur=new Terrain(new Rectangle(4.9,3.5,5.1,10));
int[] budgets={300,600,1200,2400,4800}; int[] graines={0,1,7,42,99};
var courbes = new[]{new List<double>(),new List<double>()};
foreach(int budget in budgets) {
    var lignes=new List<string>();
    foreach(bool etoile in new[]{false,true}) {
        var ratios=new List<double>();
        foreach(int g in graines) { var r=Planificateur.Plan(mur,depart,but,budget,g,etoile); if(r.Chemin!=null) { if(!CheminValide(r.Chemin,depart,but,mur)) throw new Exception("Chemin invalide"); ratios.Add(Geo.Longueur(r.Chemin)/Geo.Distance(depart,but)); } }
        ratios.Sort(); courbes[etoile ? 1 : 0].Add(ratios.Count==0 ? double.NaN : ratios[ratios.Count/2]); lignes.Add(ratios.Count==0 ? "échec (0/5)" : $"{ratios[ratios.Count/2]:F3} ({ratios.Count}/5)");
    }
    Console.WriteLine($"N={budget} : RRT {lignes[0]} ; RRT* {lignes[1]}");
}
var svg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' role='img' aria-label='Ratios médians selon budget' viewBox='0 0 620 360' width='620'><rect width='620' height='360' fill='white'/><path d='M70 25 V300 H590' fill='none' stroke='black'/><text x='220' y='345'>Budget N (échelle logarithmique)</text><text x='5' y='20'>Ratio longueur / borne</text>");
double ymin=1.0, ymax=Math.Max(1.1,courbes.SelectMany(x=>x).Where(double.IsFinite).DefaultIfEmpty(1.1).Max()+0.03);
for(int i=0;i<budgets.Length;i++) { double x=70+i*130; svg.Append($"<text x='{x-14}' y='320' font-size='12'>{budgets[i]}</text>"); }
for(int i=0;i<=4;i++) { double y=300-i*275/4.0, v=ymin+(ymax-ymin)*i/4; svg.Append($"<text x='22' y='{y+4}' font-size='12'>{v:F2}</text><path d='M70 {y} H590' stroke='#ddd'/>"); }
for(int k=0;k<2;k++) { var color=k==0?"#175aa6":"#c3422b"; for(int i=0;i<budgets.Length;i++) if(double.IsFinite(courbes[k][i])) { double x=70+i*130,y=300-(courbes[k][i]-ymin)/(ymax-ymin)*275; svg.Append($"<circle cx='{x}' cy='{y}' r='4' fill='{color}'/>"); if(i>0 && double.IsFinite(courbes[k][i-1])) { double yp=300-(courbes[k][i-1]-ymin)/(ymax-ymin)*275; svg.Append($"<path d='M{x-130} {yp} L{x} {y}' stroke='{color}' fill='none'/>"); } } }
svg.Append("<text x='380' y='22' fill='#175aa6'>RRT</text><text x='450' y='22' fill='#c3422b'>RRT*</text></svg>");
display(new PlotSvg(svg.ToString()));


N=300 : RRT 1.231 (5/5) ; RRT* 1.052 (5/5)


N=600 : RRT 1.231 (5/5) ; RRT* 1.026 (5/5)


N=1200 : RRT 1.231 (5/5) ; RRT* 1.020 (5/5)


N=2400 : RRT 1.231 (5/5) ; RRT* 1.018 (5/5)


N=4800 : RRT 1.231 (5/5) ; RRT* 1.017 (5/5)


Budget N (échelle logarithmique) Ratio longueur / borne 300 600 1200 2400 4800 1.00 1.07 1.13 1.20 1.26 RRT RRT*

### Exercice 3 — corridor

Formulez une hypothèse sur l'effet d'un corridor de largeur 0,8, 1,6 puis 3,2. Pour chaque largeur comparez les deux algorithmes sur trois graines et un budget de 1 200. Rapportez aussi le nombre d'échecs.

In [10]:
// TODO : créer les terrains et mesurer les ratios par largeur et graine.
Dictionary<double, double> ratiosRrt = null;
Dictionary<double, double> ratiosStar = null;
Console.WriteLine("Exercice 3 à compléter");

Exercice 3 à compléter


## Conclusion et références

Le reparentage peut réduire des chemins existants ; les coûts stockés doivent rester cohérents avec toute la chaîne de parents. Le balayage des voisins coûte $O(N)$ par insertion. Les résultats dépendent des graines et du terrain.

- LaValle, *Rapidly-exploring Random Trees*, 1998.
- Karaman et Frazzoli, *Sampling-based Algorithms for Optimal Motion Planning*, 2011, arXiv:1105.1186.
- [Version Python](Search-02d-RRT-Recherche-Continue.ipynb) ; [suite : recherche informée](Search-03-Informed.ipynb).